In [29]:
import aiohttp
import bs4
import json
import datetime
import asyncio
import logging
from aiogram import Bot, types
from aiogram.enums import ParseMode
from aiogram.client.bot import DefaultBotProperties
from aiogram.types import InlineKeyboardButton, InlineKeyboardMarkup

from dotenv import load_dotenv
import os
import re


In [30]:
# Загрузка конфигураций из файла .env
load_dotenv()

# Получение токена и chat_id из переменных окружения
TOKEN = os.getenv("TOKEN")
CHAT_ID = os.getenv("CHAT_ID")

In [31]:
bot = Bot(token=TOKEN, default=DefaultBotProperties(parse_mode=ParseMode.HTML))

In [32]:
async def send_media_group_or_send_message(bot, chat_id, text, gallery_urls):

    if not gallery_urls:
        await send_message(bot, chat_id, text)
    else:
        photos = [types.InputMediaPhoto(media=photo) for photo in gallery_urls[0:9]]
        photos[0].caption = text
        await bot.send_media_group(chat_id=chat_id, media=photos)


async def send_message(bot, chat_id, text):
    await bot.send_message(
        chat_id,
        text,
    )

In [33]:
headers = {
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, как Gecko) Chrome/114.0.0.0 Safari/537.36",
    "accept": "*/*",
}

In [34]:
async def get_goods(session, url="https://www.kufar.by/l"):
    async with session.get(url, headers=headers) as response:
        if response.status != 200:
            return
        kufarPage = bs4.BeautifulSoup(await response.text(), "html.parser")
        script = kufarPage.find_all("script", {"id": "__NEXT_DATA__"})
        jsonKufar = json.loads(script[0].text)
        goodsJSON = jsonKufar["props"]["initialState"]["listing"]["ads"]
        goods = []
        for g in goodsJSON:
            goods.append(
                [
                    g["subject"],
                    int(g["price_byn"]) / 100,
                    datetime.datetime.strptime(g["list_time"], "%Y-%m-%dT%H:%M:%SZ")
                    + datetime.timedelta(hours=3),
                    g["ad_link"],
                ]
            )
        return goods

In [38]:
session = aiohttp.ClientSession()
goods = await get_goods(session, 'https://re.kufar.by/l/minsk/snyat/kvartiru-dolgosrochno')
print(goods)

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x0000026C21E97A60>


[['Сдать квартиру', 933.06, datetime.datetime(2024, 7, 28, 2, 5, 30), 'https://re.kufar.by/vi/243121816'], ['Двухкомнатная квартира на полгода', 777.55, datetime.datetime(2024, 7, 28, 1, 36, 15), 'https://re.kufar.by/vi/243637092'], ['Квартира находится в чижовке Голодеда3', 750.0, datetime.datetime(2024, 7, 28, 1, 5, 45), 'https://re.kufar.by/vi/243636783'], ['Сдам однокомнатную квартиру на длительный срок', 1088.57, datetime.datetime(2024, 7, 28, 0, 34, 44), 'https://re.kufar.by/vi/243636269'], ['Сдается 1комнатная каартира', 1088.57, datetime.datetime(2024, 7, 28, 0, 11, 39), 'https://re.kufar.by/vi/243632866'], ['Квартира 1-комн', 690.0, datetime.datetime(2024, 7, 27, 23, 48, 29), 'https://re.kufar.by/vi/243632240'], ['Квартира в самом сердце столицы', 1244.08, datetime.datetime(2024, 7, 27, 23, 3, 47), 'https://re.kufar.by/vi/243626798'], ['Сдам двухкомнатную квартиру Парк Челюскинцев ', 1710.61, datetime.datetime(2024, 7, 27, 23, 2, 27), 'https://re.kufar.by/vi/243626750'], ['Сда

In [39]:
async def get_info_good(session, url):
    async with session.get(url, headers=headers) as response:
        kufarPage = bs4.BeautifulSoup(await response.text(), "html.parser")
        script = kufarPage.find_all("script", {"id": "__NEXT_DATA__"})
        jsonKufar = json.loads(script[0].text)

        return jsonKufar

In [40]:
def remove_html_tags(text):
    clean_text = re.sub(r"<[^>]+>", "", text)
    return clean_text

def parse_good(json_kufar, good, length_message=-1):
    message_max_length = 1024  

    gallery_urls = json_kufar["props"]["initialState"]["adView"]["data"]["images"][
        "gallery"
    ]
    description = json_kufar["props"]["initialState"]["adView"]["data"]["description"]
    seller_name = json_kufar["props"]["initialState"]["adView"]["data"]["userName"]
    seller_url = (
        "https://www.kufar.by/customer/"
        + json_kufar["props"]["initialState"]["adView"]["data"]["accountId"]
    )
    ad_params = json_kufar["props"]["initialState"]["adView"]["data"]["adParams"]
    params = ""
    for param in list(ad_params):
        params += f"<b>{ad_params[param]['pl']}</b>: {str(ad_params[param]['vl']).replace('<', 'меньше ').replace('>', 'больше ')}\n"
    if length_message > message_max_length:
        offset = length_message - message_max_length + 1 # one for symbol '…'
        description = description[:-offset].rstrip()
        description = description + '…' # из-за этой строчки ядро падает

    message_text = f"""
<b><a href="{good[3]}">{good[0]}</a></b>
Цена: <u>{good[1]} BYN</u>

<blockquote expandable><b>Параметры:</b>
{params}</blockquote>
<blockquote expandable><b>Описание</b>:
{description.replace('<', 'меньше ').replace('>', 'больше ')}</blockquote>

<b>Продавец</b>: <a href="{seller_url}">{seller_name.replace('<', 'меньше ').replace('>', 'больше ')}</a>
{good[2].strftime("%d.%m.%Y, %H:%M:%S")}"""
    length_message = len(remove_html_tags(message_text))
    if length_message > message_max_length:
        print('hmm')
        return parse_good(json_kufar, good, length_message)
    return gallery_urls, message_text

In [43]:
good = goods[1]
json_kufar = await get_info_good(session, good[3])
imgs, text = parse_good(json_kufar, good)

await send_media_group_or_send_message(bot, CHAT_ID, text, imgs)

hmm


In [13]:
text

'\n<b><a href="https://www.kufar.by/item/243640511">Сумка David Jones</a></b>\nЦена: <u>15.0 BYN</u>\n\n<blockquote expandable><b>Параметры:</b>\n<b>Тип оплаты</b>: Цена\n<b>Тип сделки</b>: Продажа\n<b>Категория</b>: Запчасти\n<b>Тип</b>: Запчасти для легковых авто\n<b>Запчасть</b>: Заслонка дроссельная\n<b>Марка</b>: Porsche\n<b>Год</b>: 2006\n<b>Номер запчасти</b>: \n<b>Артикул</b>: \n<b>Коробка передач</b>: Автоматическая\n<b>Тип двигателя</b>: Бензин\n<b>Состояние</b>: Б/у\n<b>Область</b>: Минск\n<b>Город / Район</b>: Центральный\n<b>Товары с Куфар Доставкой</b>: -\n<b>Товары с Куфар Оплатой</b>: -\n<b>Возможен обмен</b>: -\n</blockquote>\n<blockquote expandable><b>Описание</b>:\nЗаслонка дроссельная Porsche Cayenne 955 2006г бензин 4.5\nМаркировка: 948.605.115.03,0280750114\nХорошее состояние BOSCH. M48.00. На фото именно эта деталь, без пробега по РБ, Срок на проверку до 30 дней.\nАртикул: 999593797370</blockquote>\n\n<b>Продавец</b>: <a href="https://www.kufar.by/customer/U2FsdG